# FinReason — Setup & Data Exploration Notebook

**Project:** Teaching a Small LLM Financial Numerical Reasoning via GRPO  
**Course:** EGN 6217 — Applied Deep Learning, Spring 2026  
**Author:** Om

This notebook verifies the environment, loads the FinQA dataset, performs initial exploration, and runs the zero-shot baseline.

## 1. Environment Verification

In [ ]:
import torch
import sys
import platform

print(f"Python:     {platform.python_version()}")
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.version.cuda if torch.cuda.is_available() else 'NOT AVAILABLE'}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    print(f"VRAM:       {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

for pkg_name in ["transformers", "trl", "peft", "bitsandbytes", "accelerate", "datasets", "streamlit"]:
    try:
        mod = __import__(pkg_name)
        ver = getattr(mod, "__version__", "ok")
        print(f"  ✓ {pkg_name}: {ver}")
    except ImportError:
        print(f"  ✗ {pkg_name}: MISSING")

## 2. Load FinQA Dataset

In [ ]:
from datasets import load_dataset

print("Downloading FinQA...")
dataset = load_dataset("ibm-research/finqa", trust_remote_code=True)

for split_name in dataset:
    print(f"  {split_name}: {len(dataset[split_name])} examples")
print(f"\nColumns: {dataset['train'].column_names}")

## 3. Understand the Data Structure

In [ ]:
example = dataset["train"][0]
print("=== FIRST TRAINING EXAMPLE ===\n")
for key in example:
    val = str(example[key])
    if len(val) > 300:
        val = val[:300] + "..."
    print(f"  {key}: {val}\n")

## 4. Data Extraction Functions

In [ ]:
import sys, os, re
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
from shared_utils import extract_qa, extract_context

q, a = extract_qa(dataset["train"][0])
ctx = extract_context(dataset["train"][0])
print(f"Question: {q}")
print(f"Answer:   {a}")
print(f"Context:  {len(ctx)} characters")

## 5. Answer Distribution Analysis

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

train = dataset["train"]
answer_types = Counter()
answer_lengths, context_lengths = [], []

for ex in train:
    _, a = extract_qa(ex)
    ctx = extract_context(ex)
    answer_lengths.append(len(a.strip()))
    context_lengths.append(len(ctx))
    cleaned = re.sub(r'[,$%]', '', a.strip())
    try:
        float(cleaned)
        answer_types["numeric"] += 1
    except ValueError:
        if a.strip().lower() in ("yes", "no"):
            answer_types["yes/no"] += 1
        else:
            answer_types["text/span"] += 1

print(f"Total: {len(train)}")
for t, c in answer_types.most_common():
    print(f"  {t}: {c} ({100*c/len(train):.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

types_list = list(answer_types.keys())
counts_list = list(answer_types.values())
axes[0].bar(types_list, counts_list, color=["#3b82f6","#10b981","#f59e0b"][:len(types_list)], edgecolor="black", lw=0.5)
axes[0].set_title("Answer Types"); axes[0].set_ylabel("Count")

axes[1].hist(context_lengths, bins=50, color="#3b82f6", edgecolor="black", lw=0.3)
axes[1].set_title("Context Length (chars)"); axes[1].axvline(np.mean(context_lengths), color="red", ls="--")

axes[2].hist(answer_lengths, bins=30, color="#10b981", edgecolor="black", lw=0.3)
axes[2].set_title("Answer Length (chars)"); axes[2].axvline(np.mean(answer_lengths), color="red", ls="--")

plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/data_exploration.png", dpi=150)
plt.show()
print("Saved to results/data_exploration.png")

## 6. Sample Examples

In [ ]:
import random
random.seed(42)
for idx in random.sample(range(len(train)), 3):
    ex = train[idx]
    q, a = extract_qa(ex)
    ctx = extract_context(ex)
    print(f"--- Example {idx} ---")
    print(f"  Q: {q}")
    print(f"  A: {a}")
    if "qa" in ex and isinstance(ex["qa"], dict):
        print(f"  Program: {ex['qa'].get('program', 'N/A')}")
    print()

## 7. Reward Function Self-Test

In [ ]:
from shared_utils import extract_final_answer, check_answer, reward_function

tests = [
    ("42.5", "42.5", True), ("$1,452.4", "1452.4", True),
    ("1.45B", "1450000000", True), ("(3.5)", "-3.5", True),
    ("45.2%", "45.2", True), ("50", "42", False),
    ("<think>306.2</think>\n306.2", "306.2", True),
    ("Answer: 306.2", "306.2", True),
]

passed = 0
for out, gt, exp in tests:
    final = extract_final_answer(out)
    ok = check_answer(final, gt) == exp
    passed += ok
    r = reward_function(out, gt)
    print(f"  {'✓' if ok else '✗'} '{out[:40]:40s}' vs '{gt:15s}' R={r:.1f}")
print(f"\n  {passed}/{len(tests)} passed")

## 8. Model Loading Test (4-bit)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {MODEL_NAME} in 4-bit...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True),
    device_map="auto", trust_remote_code=True)
model.eval()

print(f"  VRAM used: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 9. Quick Zero-Shot Test (5 examples)

In [ ]:
from shared_utils import format_prompt

test_data = dataset["test"] if "test" in dataset else dataset["validation"]

correct_count = 0
for i in range(5):
    q, gt = extract_qa(test_data[i])
    ctx = extract_context(test_data[i])
    prompt = format_prompt(ctx, q, answer=None)
    ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    pred = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    final = extract_final_answer(pred)
    is_correct = check_answer(final, gt)
    correct_count += is_correct
    print(f"  {'✓' if is_correct else '✗'} GT={gt} | Pred={final}")

print(f"\nQuick check: {correct_count}/5 correct")
del model; torch.cuda.empty_cache()

## 10. Summary

- Dataset loaded and explored ✓
- Reward function tested ✓
- Model loads in 4-bit ✓
- Zero-shot inference works ✓

**Next step:** Run the full training pipeline (SFT → GRPO).